### Ministério da saúde (SUS) - Vacinação

Baixa as partes nacionais oficiais de vacinacao COVID-19 e salva Parquet.

Notebook:
    %pip install pandas pyarrow
    %run baixar_vacinacao_covid_2021_2024.py

Saida: dados_vacinacao_covid/parquet/vacinacao_covid_parte_*.parquet
Periodo: data de aplicacao >= 2021-01-01 e < 2025-01-01.
Fonte exclusiva COVID-19: nao mistura com o PNI geral.
Cada arquivo nacional pode conter varios anos. A filtragem ocorre apos baixar
cada parte. Este download pode ser muito maior e mais demorado que SRAG.
Processamento em blocos; todas as colunas mantidas como texto.
Datas vazias/invalidas vao a quarentena; nao entram no recorte temporal.
Nao deduplica registros nem calcula cobertura vacinal.

Em 22/10/2026 o catalogo oficial retornou 502/503 neste ambiente. A descoberta
de links e obrigatoria e pode falhar enquanto o portal estiver indisponivel.
O fluxo de conversao foi testado com dados simulados, nao com a base nacional.
Arquivos Parquet concluidos e registrados no manifesto sao reutilizados.
Uma interrupcao reinicia apenas a parte incompleta. Nao execute duas copias
simultaneamente na mesma pasta. Arquivos brutos baixados por este script sao
removidos apos conversao bem-sucedida, se REMOVER_CSV_APOS_CONVERTER=True.

In [1]:
import csv
import hashlib
import html
import io
import json
import re
import time
from datetime import datetime, timezone
from pathlib import Path
from urllib.parse import urlsplit
from urllib.request import Request, urlopen

import pandas as pd
import pyarrow as pa
import pyarrow.parquet as pq

In [2]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_vacinacao_covid'
PAGINA = ('https://dadosabertos.saude.gov.br/dataset/covid-19-vacinacao/resource/'
          '301983f2-aa50-4977-8fec-cfab0806cb0b')
LOTE = 50000
ENCODING = 'utf-8-sig'
REMOVER_CSV_APOS_CONVERTER = True
INICIO = pd.Timestamp('2021-01-01',tz='UTC')
FIM = pd.Timestamp('2025-01-01',tz='UTC')

In [3]:
def escrever_json(caminho, dados):
    tmp = caminho.with_suffix(caminho.suffix+'.part')
    tmp.write_text(json.dumps(dados,ensure_ascii=False,indent=2),encoding='utf-8')
    tmp.replace(caminho)


def requisitar(url):
    return urlopen(Request(url,headers={'User-Agent':'Observatorio-COVID-Brasil/1.0',
                                        'Accept-Encoding':'identity'}),timeout=120)


def extrair_links(conteudo):
    texto = html.unescape(conteudo).replace('\\/','/').replace('\\u0026','&')
    urls = sorted(set(re.findall(r'https?://[^\s<>"\x27\\]+?\.csv(?:\?[^\s<>"\x27\\]*)?',texto,re.I)))
    # Aceita apenas arquivos divulgados no recurso oficial.
    urls = [u for u in urls if urlsplit(u).scheme=='https']
    partes = set(re.findall(r'Dados\s+Completos\s*[-–—]\s*Parte\s*(\d+)',texto,re.I))
    if not urls:
        raise ValueError('Nenhum link CSV encontrado no recurso oficial. Nao foi iniciado o download.')
    if not partes or len(partes)!=len(urls):
        raise ValueError(f'Nao foi possivel confirmar a lista completa: {len(partes)} partes rotuladas e {len(urls)} links CSV. O portal pode ter mudado.')
    return urls


def descobrir_links():
    cache=PASTA/'catalogo_links.json'
    if cache.exists():
        obj=json.loads(cache.read_text(encoding='utf-8'))
        if obj.get('pagina')!=PAGINA or not obj.get('urls'):
            raise ValueError('Cache de catalogo invalido.')
        print('Usando lista de fontes preservada da primeira execucao.',flush=True)
        return obj['urls']
    for tentativa in range(1,4):
        try:
            print(f'Consultando catalogo oficial ({tentativa}/3)...',flush=True)
            with requisitar(PAGINA) as r:
                texto=r.read().decode('utf-8')
            urls=extrair_links(texto)
            escrever_json(cache,{'pagina':PAGINA,'consulta_utc':datetime.now(timezone.utc).isoformat(),'urls':urls})
            return urls
        except Exception as e:
            print(f'Catalogo: {e}',flush=True)
            if tentativa==3:
                raise RuntimeError('Catalogo indisponivel ou formato alterado. Execute novamente mais tarde; nenhum link foi inventado.') from e
            time.sleep(3*tentativa)


def baixar(url, destino):
    if destino.exists():
        print('Reutilizando CSV completo: '+destino.name,flush=True)
        return
    parcial=destino.with_suffix('.csv.part')
    for tentativa in range(1,4):
        try:
            recebido=0;inicio=time.monotonic();ultimo=inicio
            with requisitar(url) as resposta, parcial.open('wb') as f:
                if resposta.status!=200:
                    raise IOError(f'HTTP inesperado: {resposta.status}')
                total=int(resposta.headers.get('Content-Length',0))
                if 'text/html' in resposta.headers.get('Content-Type','').lower():
                    raise ValueError('Servidor retornou HTML em vez de CSV.')
                while True:
                    bloco=resposta.read(4*1024*1024)
                    if not bloco:break
                    f.write(bloco);recebido+=len(bloco)
                    agora=time.monotonic()
                    if agora-ultimo>=3:
                        porcentagem=f'{100*recebido/total:.1f}%' if total else 'total desconhecido'
                        print(f'  {recebido/1024**3:.2f} GiB | {porcentagem} | {recebido/max(agora-inicio,.001)/1024**2:.1f} MiB/s',flush=True)
                        ultimo=agora
            if not recebido or (total and recebido!=total):
                raise IOError('Download vazio ou incompleto.')
            parcial.replace(destino)
            return
        except Exception as e:
            print(f'Falha no download ({tentativa}/3): {e}',flush=True)
            if tentativa==3:raise
            time.sleep(2*tentativa)


def converter(origem, destino, rejeitados):
    with origem.open(encoding=ENCODING,newline='') as f:
        cabecalho=f.readline()
    separador=csv.Sniffer().sniff(cabecalho,delimiters=';,\t').delimiter
    nomes=next(csv.reader([cabecalho],delimiter=separador))
    if not nomes or len(nomes)!=len(set(nomes)):
        raise ValueError('Cabecalho vazio ou com colunas repetidas.')
    normalizar=lambda s: re.sub(r'[^a-z0-9]','',s.lower())
    candidatas=[c for c in nomes if normalizar(c)=='vacinadataaplicacao']
    if len(candidatas)!=1:
        raise ValueError('Campo vacina_dataAplicacao nao encontrado. Colunas: '+str(nomes))
    coluna_data=candidatas[0]
    esquema=pa.schema([(c,pa.string()) for c in nomes])
    parcial=destino.with_suffix('.parquet.part')
    quarentena=rejeitados.with_suffix('.parquet.part')
    total=selecionados=invalidos=fora=0
    # Os dois writers garantem saidas validas mesmo para um recorte vazio.
    with pq.ParquetWriter(parcial,esquema,compression='snappy') as w, pq.ParquetWriter(quarentena,esquema,compression='snappy') as q:
        leitor=pd.read_csv(origem,sep=separador,encoding=ENCODING,dtype='string',
            chunksize=LOTE,keep_default_na=False,on_bad_lines='error')
        for lote in leitor:
            bruto=lote[coluna_data].str.strip()
            datas=pd.to_datetime(bruto,format='ISO8601',errors='coerce',utc=True)
            ausente=datas.isna()
            if ausente.any():
                alternativa=pd.to_datetime(bruto.where(ausente),format='%d/%m/%Y',errors='coerce',utc=True)
                datas=datas.fillna(alternativa)
            invalida=datas.isna()
            dentro=(datas>=INICIO)&(datas<FIM)
            if dentro.any():
                w.write_table(pa.Table.from_pandas(lote.loc[dentro],schema=esquema,preserve_index=False))
            if invalida.any():
                q.write_table(pa.Table.from_pandas(lote.loc[invalida],schema=esquema,preserve_index=False))
            total+=len(lote);selecionados+=int(dentro.sum());invalidos+=int(invalida.sum())
            fora+=int((~dentro & ~invalida).sum())
            print(f'  Lidos: {total:,} | 2021–2024: {selecionados:,} | datas invalidas/ausentes: {invalidos:,}',flush=True)
    if total==0:raise ValueError('CSV sem registros; parte nao concluida.')
    if total!=selecionados+invalidos+fora:raise RuntimeError('Contagens divergentes.')
    if pq.read_metadata(parcial).num_rows!=selecionados:raise RuntimeError('Parquet incompleto.')
    parcial.replace(destino);quarentena.replace(rejeitados)
    return {'lidos':total,'selecionados_2021_2024':selecionados,'datas_invalidas_ou_ausentes':invalidos,
            'fora_periodo':fora,'colunas':len(nomes)}


def main():
    for sub in ['brutos','parquet','quarentena']:(PASTA/sub).mkdir(parents=True,exist_ok=True)
    links=descobrir_links()
    print(f'{len(links)} partes nacionais. O filtro temporal sera aplicado depois de cada download.',flush=True)
    manifesto=PASTA/'manifesto_download.json'
    estados=json.loads(manifesto.read_text(encoding='utf-8')) if manifesto.exists() else {}
    for i,url in enumerate(links,1):
        chave=hashlib.sha256(url.encode()).hexdigest()[:16]
        destino=PASTA/'parquet'/f'vacinacao_covid_parte_{chave}.parquet'
        rejeitados=PASTA/'quarentena'/f'datas_invalidas_{chave}.parquet'
        origem=PASTA/'brutos'/f'parte_{chave}.csv'
        anterior=estados.get(chave,{})
        if anterior.get('status')=='concluido' and anterior.get('url')==url and destino.exists() and rejeitados.exists():
            if pq.read_metadata(destino).num_rows==anterior['selecionados_2021_2024'] and pq.read_metadata(rejeitados).num_rows==anterior['datas_invalidas_ou_ausentes']:
                print(f'Parte {i}/{len(links)} ja concluida.',flush=True);continue
        print(f'\nParte {i}/{len(links)}: {url}',flush=True)
        try:
            baixar(url,origem)
            info=converter(origem,destino,rejeitados)
            estados[chave]={'status':'concluido','url':url,'arquivo':str(destino),
                'concluido_utc':datetime.now(timezone.utc).isoformat(),**info}
            escrever_json(manifesto,estados)
            if REMOVER_CSV_APOS_CONVERTER:origem.unlink()
        except Exception as e:
            estados[chave]={'status':'erro','url':url,'erro':str(e)}
            escrever_json(manifesto,estados)
            raise RuntimeError('Coleta incompleta. Detalhes no manifesto. Execute novamente para retomar.') from e
    print('\nDownload e conversao das partes concluidos. Saida: '+str(PASTA/'parquet'))
    print('Verifique as contagens por parte no manifesto e a quarentena antes da analise.')
    return estados


if __name__=='__main__':
    resultado_vacinacao=main()


Consultando catalogo oficial (1/3)...
Catalogo: [WinError 10054] Foi forçado o cancelamento de uma conexão existente pelo host remoto
Consultando catalogo oficial (2/3)...
Catalogo: [WinError 10054] Foi forçado o cancelamento de uma conexão existente pelo host remoto
Consultando catalogo oficial (3/3)...
Catalogo: [WinError 10054] Foi forçado o cancelamento de uma conexão existente pelo host remoto


RuntimeError: Catalogo indisponivel ou formato alterado. Execute novamente mais tarde; nenhum link foi inventado.

### Análise exploratória

In [4]:
import base64
import hashlib
import html
import json
import re
from datetime import datetime
from pathlib import Path

import duckdb
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pyarrow.parquet as pq

In [5]:
BASE = Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()
PASTA = BASE / 'dados_vacinacao_covid'
SAIDA = PASTA / 'analise'
MEMORIA = '2GB'
THREADS = 2
PERMITIR_COLETA_PARCIAL = False
# Estas contagens exatas podem ser demoradas na base nacional.
VERIFICAR_IDS = False

In [ ]:
CAMPOS = {
 'data':['vacina_dataAplicacao'],
 'vacina':['vacina_nome'],
 'fabricante':['vacina_fabricante_nome'],
 'dose':['vacina_descricao_dose','vacina_descricaoDose'],
 'uf_residencia':['paciente_endereco_uf'],
 'uf_aplicacao':['estabelecimento_uf','estabelecimento_uf_nome'],
 'sexo':['paciente_enumSexoBiologico'],
 'idade':['paciente_idade'],
 'grupo':['vacina_grupoAtendimento_nome'],
 'documento':['document_id'],
 'paciente':['paciente_id'],
}


def lit(x):return "'"+str(x).replace("'","''")+"'"
def ident(x):return '"'+str(x).replace('"','""')+'"'
def norm(x):return re.sub('[^a-z0-9]','',x.lower())


def checar_coleta(arquivos):
    catalogo=PASTA/'catalogo_links.json';manifesto=PASTA/'manifesto_download.json'
    if not catalogo.exists() or not manifesto.exists():
        return 'COMPLETUDE NAO VERIFICADA: catalogo ou manifesto ausente.',[]
    urls=json.loads(catalogo.read_text(encoding='utf-8')).get('urls',[])
    estados=json.loads(manifesto.read_text(encoding='utf-8'))
    if not urls or len(urls)!=len(set(urls)):
        raise ValueError('Catalogo vazio ou com URLs duplicadas.')
    esperados=set();problemas=[];info=[]
    for url in urls:
        chave=hashlib.sha256(url.encode()).hexdigest()[:16]
        nome=f'vacinacao_covid_parte_{chave}.parquet';esperados.add(nome)
        e=estados.get(chave,{})
        caminho=PASTA/'parquet'/nome
        ok=e.get('status')=='concluido' and e.get('url')==url and caminho.exists()
        if ok:
            ok=pq.read_metadata(caminho).num_rows==e.get('selecionados_2021_2024')
        if not ok:problemas.append(nome)
        info.append({'arquivo':nome,'conferido':ok,'lidos_na_origem':e.get('lidos'),
            'selecionados_2021_2024':e.get('selecionados_2021_2024'),
            'datas_invalidas_ou_ausentes_na_origem':e.get('datas_invalidas_ou_ausentes'),
            'fora_periodo_na_origem':e.get('fora_periodo')})
    extras=set(p.name for p in arquivos)-esperados
    if extras:raise ValueError('Partes fora do catalogo: '+', '.join(sorted(extras)))
    if problemas and not PERMITIR_COLETA_PARCIAL:
        raise ValueError('Download incompleto ou contagens divergentes. Termine o download primeiro. Partes: '+', '.join(problemas))
    return ('COLETA PARCIAL: faltam partes ou ha contagens divergentes.' if problemas else
            'Arquivos e contagens conferidos com o catalogo salvo. Isso nao certifica completude epidemiologica da fonte.'),info


def main():
    arquivos=sorted((PASTA/'parquet').glob('vacinacao_covid_parte_*.parquet'))
    if not arquivos:raise FileNotFoundError('Nenhuma parte Parquet encontrada. Conclua o download em '+str(PASTA/'parquet'))
    SAIDA.mkdir(parents=True,exist_ok=True)
    status,manifesto=checar_coleta(arquivos)
    print(status,flush=True)
    con=duckdb.connect()
    con.execute('SET memory_limit='+lit(MEMORIA));con.execute(f'SET threads={THREADS}')
    con.execute('SET temp_directory='+lit(SAIDA/'_temporarios'))
    con.execute('SET preserve_insertion_order=false')
    secoes=[];resultados={}
    def exportar(df,nome):
        df.to_csv(SAIDA/(nome+'.csv'),index=False,sep=';',decimal=',',encoding='utf-8-sig')
        df.to_parquet(SAIDA/(nome+'.parquet'),index=False)
    def mostrar(titulo,df,limite=35):
        return '<section><h2>'+html.escape(titulo)+'</h2><div class="scroll">'+df.head(limite).to_html(index=False,border=0,escape=True,na_rep='—')+'</div>'+('<p>Tabela completa em CSV e Parquet.</p>' if len(df)>limite else '')+'</section>'
    def consulta(nome,titulo,sql):
        print('Analisando: '+titulo,flush=True)
        df=con.execute(sql).fetchdf();exportar(df,nome);resultados[nome]=df
        secoes.append(mostrar(titulo,df));return df
    def figura(fig,nome,legenda):
        fig.tight_layout();p=SAIDA/(nome+'.png');fig.savefig(p,dpi=130,bbox_inches='tight');plt.close(fig)
        return '<section><img alt="'+html.escape(legenda)+'" src="data:image/png;base64,'+base64.b64encode(p.read_bytes()).decode()+'"><p>'+html.escape(legenda)+'</p></section>'
    try:
        print('Conferindo colunas de cada parte...',flush=True)
        consultas=[];inventario=[];mapeamento=[]
        for p in arquivos:
            meta=pq.read_metadata(p)
            nomes=pq.ParquetFile(p).schema_arrow.names
            disponiveis={}
            for n in nomes:
                if norm(n) in disponiveis:raise ValueError('Colunas ambiguas: '+p.name)
                disponiveis[norm(n)]=n
            selecao=[]
            for alias,candidatas in CAMPOS.items():
                achadas=list(dict.fromkeys(disponiveis[norm(c)] for c in candidatas if norm(c) in disponiveis))
                if len(achadas)>1:raise ValueError(f'Campo ambiguo {alias} em {p.name}: {achadas}')
                original=achadas[0] if achadas else None
                if alias=='data' and original is None:raise ValueError('Coluna de data de aplicacao ausente em '+p.name)
                expr='NULL::VARCHAR' if original is None else 'NULLIF(TRIM(CAST('+ident(original)+" AS VARCHAR)), '')"
                selecao.append(expr+' AS '+ident(alias))
                mapeamento.append({'arquivo':p.name,'campo_analitico':alias,'coluna_origem':original})
            selecao.append(lit(p.name)+' AS arquivo_origem')
            consultas.append('SELECT '+','.join(selecao)+' FROM read_parquet('+lit(p)+')')
            inventario.append({'arquivo':p.name,'registros':meta.num_rows,'colunas':meta.num_columns,'tamanho_mib':round(p.stat().st_size/1024**2,2)})
        exportar(pd.DataFrame(mapeamento),'mapeamento_colunas')
        exportar(pd.DataFrame(inventario),'arquivos_entrada')
        if manifesto:
            exportar(pd.DataFrame(manifesto),'conferencia_download')
            secoes.append(mostrar('Qualidade na origem — contagens do download',pd.DataFrame(manifesto)))
        con.execute('CREATE VIEW origem AS '+ ' UNION ALL '.join(consultas))
        # Para ISO com fuso horario, respeita o instante UTC usado no download.
        con.execute("SET TimeZone='UTC'")
        con.execute('''CREATE VIEW preparada AS SELECT *,
          coalesce(try_cast(data AS TIMESTAMPTZ)::DATE,
                   try_strptime(data,'%d/%m/%Y')::DATE) AS data_aplicacao,
          try_cast(idade AS DOUBLE) AS idade_numerica FROM origem''')
        qualidade=consulta('qualidade_datas','Datas e recorte temporal', '''SELECT count(*) AS registros_lidos,
          count(*) FILTER(WHERE data IS NULL) AS datas_ausentes,
          count(*) FILTER(WHERE data IS NOT NULL AND data_aplicacao IS NULL) AS datas_invalidas,
          count(*) FILTER(WHERE data_aplicacao < DATE '2021-01-01' OR data_aplicacao >= DATE '2025-01-01') AS fora_periodo,
          count(*) FILTER(WHERE data_aplicacao >= DATE '2021-01-01' AND data_aplicacao < DATE '2025-01-01') AS no_periodo
          FROM preparada''')
        con.execute('''CREATE VIEW analise AS SELECT *,
          CASE WHEN idade IS NULL THEN 'Ausente'
               WHEN idade_numerica IS NULL OR NOT isfinite(idade_numerica) OR idade_numerica<0 OR idade_numerica>150 OR idade_numerica!=floor(idade_numerica) THEN 'Invalida'
               WHEN idade_numerica<5 THEN '00: 0–4 anos'
               WHEN idade_numerica<12 THEN '01: 5–11 anos'
               WHEN idade_numerica<18 THEN '02: 12–17 anos'
               WHEN idade_numerica<40 THEN '03: 18–39 anos'
               WHEN idade_numerica<60 THEN '04: 40–59 anos'
               WHEN idade_numerica<80 THEN '05: 60–79 anos'
               ELSE '06: 80–150 anos' END AS faixa_etaria
          FROM preparada WHERE data_aplicacao >= DATE '2021-01-01' AND data_aplicacao < DATE '2025-01-01' ''')
        if int(qualidade.iloc[0]['no_periodo'])==0:raise ValueError('Nao ha registros validos em 2021–2024. Confira qualidade_datas.csv.')
        consulta('resumo_geral','Visao geral dos registros de aplicacao', '''SELECT count(*) AS registros_aplicacao,
          min(data_aplicacao) AS primeira_aplicacao, max(data_aplicacao) AS ultima_aplicacao,
          count(DISTINCT vacina) AS nomes_vacinas, count(DISTINCT dose) AS descricoes_doses,
          count(DISTINCT arquivo_origem) AS partes_com_registros_no_periodo FROM analise''')
        consulta('registros_por_ano','Aplicacoes por ano', '''SELECT year(data_aplicacao) AS ano,count(*) AS registros
          FROM analise GROUP BY 1 ORDER BY 1''')
        consulta('registros_por_mes','Aplicacoes por mes', '''WITH meses AS (
          SELECT unnest(generate_series(DATE '2021-01-01',DATE '2024-12-01',INTERVAL 1 MONTH)) AS mes),
          contagem AS (SELECT date_trunc('month',data_aplicacao) AS mes,count(*) AS registros FROM analise GROUP BY 1)
          SELECT strftime(m.mes,'%Y-%m') AS mes,coalesce(c.registros,0) AS registros
          FROM meses m LEFT JOIN contagem c USING(mes) ORDER BY 1''')
        for campo,titulo in [('vacina','Vacinas — nomes na fonte'),('fabricante','Fabricantes'),('dose','Doses — descricoes na fonte'),
            ('uf_residencia','UF de residencia'),('uf_aplicacao','UF do estabelecimento de aplicacao'),
            ('sexo','Sexo — codigos originais'),('faixa_etaria','Faixas etarias por registro'),('grupo','Grupo de atendimento')]:
            expr='upper('+ident(campo)+')' if campo.startswith('uf_') else ident(campo)
            consulta('distribuicao_'+campo,titulo,f'''SELECT coalesce({expr},'Ausente') AS categoria,count(*) AS registros,
              round(100.0*count(*)/sum(count(*)) OVER(),2) AS percentual_registros FROM analise GROUP BY 1 ORDER BY registros DESC,categoria''')
        consulta('ano_uf_residencia','Aplicacoes por ano e UF de residencia', '''SELECT year(data_aplicacao) AS ano,
           coalesce(upper(uf_residencia),'Ausente') AS uf,count(*) AS registros FROM analise GROUP BY 1,2 ORDER BY 1,2''')
        consulta('ano_dose','Aplicacoes por ano e descricao de dose', '''SELECT year(data_aplicacao) AS ano,
           coalesce(dose,'Ausente') AS dose,count(*) AS registros FROM analise GROUP BY 1,2 ORDER BY 1,2''')
        campos=list(CAMPOS)
        nulos=con.execute('SELECT count(*),'+','.join('count(*) FILTER(WHERE '+ident(c)+' IS NULL)' for c in campos)+' FROM analise').fetchone()
        df_nulos=pd.DataFrame([{'campo':c,'ausentes':nulos[i+1],'percentual_ausente':round(100*nulos[i+1]/nulos[0],2)} for i,c in enumerate(campos)])
        exportar(df_nulos,'ausencias_campos');secoes.append(mostrar('Ausencias nos campos analisados — NULL ou texto vazio',df_nulos))
        consulta('qualidade_idade','Qualidade da idade informada', '''SELECT faixa_etaria,count(*) AS registros
          FROM analise WHERE faixa_etaria IN ('Ausente','Invalida') GROUP BY 1''')
        if VERIFICAR_IDS:
            consulta('identificadores','Identificadores preenchidos — sem deduplicacao automatica', '''SELECT
              count(documento) AS registros_com_document_id,count(DISTINCT documento) AS document_ids_distintos,
              count(documento)-count(DISTINCT documento) AS excedentes_por_document_id,
              count(paciente) AS registros_com_paciente_id,count(DISTINCT paciente) AS paciente_ids_distintos
              FROM analise''')
        plt.rcParams.update({'font.family':'DejaVu Sans','axes.spines.top':False,'axes.spines.right':False})
        imagens=[]
        mensal=resultados['registros_por_mes'];fig,ax=plt.subplots(figsize=(11,4))
        ax.plot(range(len(mensal)),mensal['registros']/1e6,color='#147d92')
        ax.set_xticks(range(0,len(mensal),4),mensal['mes'].iloc[::4],rotation=45)
        ax.set(ylabel='Milhoes de registros',title='Registros de aplicacao por mes')
        imagens.append(figura(fig,'01_aplicacoes_mensais','Meses sem registros na coleta aparecem como zero; isso nao confirma ausencia de vacinacao.'))
        for nome,titulo in [('uf_residencia','UF de residencia'),('vacina','Vacinas'),('dose','Doses')]:
            d=resultados['distribuicao_'+nome].head(15).iloc[::-1]
            fig,ax=plt.subplots(figsize=(11,6));ax.barh(d['categoria'],d['registros']/1e6,color='#147d92')
            ax.set(xlabel='Milhoes de registros',title=titulo+' — ate 15 categorias mais frequentes')
            imagens.append(figura(fig,'grafico_'+nome,'Contagens de registros sem deduplicacao; categorias completas nas tabelas.'))
        nota='''<section class="nota"><h2>Como interpretar</h2><p>Uma linha representa um registro de aplicacao.
        Uma pessoa pode ter varias doses. Totais e percentuais nao representam pessoas vacinadas, esquema completo ou cobertura vacinal.
        Nenhuma duplicata foi removida.</p><p>Residencia do paciente e local de aplicacao sao dimensoes diferentes.
        Faixa etaria usa paciente_idade em anos, como informada; nao recalcula idade nem deduz data de nascimento.
        Descricoes de doses, vacinas e codigos de sexo foram preservados para revisao antes de agrupamentos epidemiologicos.</p>
        <p>As ausencias incluem campos opcionais que nao existem em alguma parte. Veja mapeamento_colunas.
        Registros com datas invalidas separados pelo download nao estao no Parquet analisado: as contagens originais, quando disponiveis,
        aparecem na conferencia do download. O recorte analitico considera data de aplicacao em 2021–2024.</p>
        <p>Uma verificacao por document_id identifica repeticoes do identificador, nao duplicatas exatas de todas as colunas.
        IDs distintos de paciente nao garantem pessoas unicas. Nenhum identificador individual aparece nas tabelas exportadas.</p></section>'''
        if not VERIFICAR_IDS:nota+='<section><p>Contagens exatas de IDs e repeticoes nao executadas. Para habilitar, defina VERIFICAR_IDS = True; essa etapa pode demorar na base nacional.</p></section>'
        pagina='''<!doctype html><html lang="pt-BR"><head><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">
        <title>Exploracao da vacinacao COVID-19</title><style>body{font-family:Arial,sans-serif;background:#eef3f7;color:#20384a;max-width:1200px;margin:auto;padding:26px}
        section{background:white;padding:22px;border-radius:12px;margin:22px 0}.nota{border-left:5px solid #d29a36}p{line-height:1.6}
        .scroll{overflow:auto}table{border-collapse:collapse;width:100%;font-size:13px}td,th{padding:9px;border-bottom:1px solid #dce5ec;text-align:left}
        th{background:#e2edf3}img{width:100%;height:auto}</style></head><body><h1>Observatorio COVID-19 Brasil</h1>
        <p>Analise exploratoria da vacinacao · 2021–2024</p>'''
        pagina+='<section class="nota"><strong>'+html.escape(status)+'</strong></section>'
        pagina+='<p>Gerado em '+datetime.now().astimezone().isoformat(timespec='seconds')+'</p>'
        pagina+=nota+''.join(imagens)+''.join(secoes)+'</body></html>'
        destino=SAIDA/'relatorio_vacinacao_covid.html';destino.write_text(pagina,encoding='utf-8')
        print('\nConcluido. Relatorio: '+str(destino),flush=True)
        return resultados
    finally:con.close()


if __name__=='__main__':
    resultados_vacinacao=main()
